In [45]:
"""
🎯 LEARNING OBJECTIVES:
- Perform time series analysis on sensor data
- Calculate correlations between different sensor types
- Engineer features for predictive modeling
- Implement window functions for trend analysis

📅 SCHEDULE:
Morning (4 hours):
1. Temporal Pattern Analysis (2 hours)
2. Cross-Sensor Correlation Analysis (2 hours)

Afternoon (4 hours):
3. Feature Engineering (3 hours)
4. Trend Analysis (1 hour)

✅ DELIVERABLES:
- Time series analysis dashboard
- Correlation study findings
- Feature engineering pipeline
- Trend analysis reports
"""

'\n🎯 LEARNING OBJECTIVES:\n- Perform time series analysis on sensor data\n- Calculate correlations between different sensor types\n- Engineer features for predictive modeling\n- Implement window functions for trend analysis\n\n📅 SCHEDULE:\nMorning (4 hours):\n1. Temporal Pattern Analysis (2 hours)\n2. Cross-Sensor Correlation Analysis (2 hours)\n\nAfternoon (4 hours):\n3. Feature Engineering (3 hours)\n4. Trend Analysis (1 hour)\n\n✅ DELIVERABLES:\n- Time series analysis dashboard\n- Correlation study findings\n- Feature engineering pipeline\n- Trend analysis reports\n'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Machine learning imports
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.linalg import Vectors

# =============================================================================
# LOAD CLEANED DATA FROM DAY 2
# =============================================================================

In [47]:
# Quick data overview
for name, df in datasets.items():
    if df is not None:
        count = df.count()
        print(f"   📊 {name}: {count:,} records")

✅ Using existing Spark session
📈 Day 3: Time Series Analysis & Feature Engineering


In [6]:
print("\n" + "=" * 60)
print("⏰ SECTION 1: TEMPORAL PATTERN ANALYSIS")
print("=" * 60)


⏰ SECTION 1: TEMPORAL PATTERN ANALYSIS


In [48]:
"""
🎯 TASK: Decompose time series into trend, seasonal, and residual components
💡 HINT: Look for daily, weekly, and monthly patterns
📚 CONCEPTS: Seasonality, trends, cyclical patterns, decomposition
"""

def analyze_temporal_patterns(df, value_col, time_col="timestamp", sensor_col=None):
    """
    Analyze temporal patterns in sensor data
    
    Args:
        df: DataFrame with time series data
        value_col: Column containing values to analyze
        time_col: Timestamp column
        sensor_col: Sensor ID column (optional)
    
    Returns:
        DataFrame with temporal pattern analysis
    """
    print(f"\n📈 Temporal Pattern Analysis: {value_col}")
    print("-" * 40)
    
    # TODO: Add time-based features for pattern analysis
    df_with_time = df.withColumn("year", F.year(time_col)) \
                     .withColumn("month", F.month(time_col)) \
                     .withColumn("day", F.dayofmonth(time_col)) \
                     .withColumn("hour", F.hour(time_col)) \
                     .withColumn("day_of_week", F.dayofweek(time_col)) \
                     .withColumn("week_of_year", F.weekofyear(time_col)) \
                     .withColumn("is_weekend", F.when(F.dayofweek(time_col).isin([1, 7]), True).otherwise(False))
    
    # TODO: Hourly patterns
    print("🕐 Hourly Patterns:")
    hourly_patterns = df_with_time.groupBy("hour").agg(
        F.avg(value_col).alias("avg_value"),
        F.stddev(value_col).alias("stddev_value"),
        F.min(value_col).alias("min_value"),
        F.max(value_col).alias("max_value"),
        F.count(value_col).alias("count_readings")
    ).orderBy("hour")
    
    hourly_patterns.show(24)
    
    # TODO: Find peak and off-peak hours
    peak_hours = hourly_patterns.orderBy(F.desc("avg_value")).limit(3)
    print("   🔝 Peak hours:")
    peak_hours.show()
    
    # TODO: Day of week patterns
    print("\n📅 Day of Week Patterns:")
    daily_patterns = df_with_time.groupBy("day_of_week").agg(
        F.avg(value_col).alias("avg_value"),
        F.count(value_col).alias("count_readings")
    ).orderBy("day_of_week")
    
    # Add day names for better readability
    daily_patterns = daily_patterns.withColumn(
        "day_name",
        F.when(F.col("day_of_week") == 1, "Sunday")
         .when(F.col("day_of_week") == 2, "Monday")
         .when(F.col("day_of_week") == 3, "Tuesday")
         .when(F.col("day_of_week") == 4, "Wednesday")
         .when(F.col("day_of_week") == 5, "Thursday")
         .when(F.col("day_of_week") == 6, "Friday")
         .when(F.col("day_of_week") == 7, "Saturday")
    )
    
    daily_patterns.select("day_name", "avg_value", "count_readings").show()
    
    # TODO: Weekend vs Weekday comparison
    weekend_vs_weekday = df_with_time.groupBy("is_weekend").agg(
        F.avg(value_col).alias("avg_value"),
        F.count(value_col).alias("count_readings")
    )
    
    print("\n🏖️ Weekend vs Weekday:")
    weekend_vs_weekday.show()
    
    # TODO: Monthly patterns (seasonal trends)
    print("\n📊 Monthly Patterns:")
    monthly_patterns = df_with_time.groupBy("month").agg(
        F.avg(value_col).alias("avg_value"),
        F.count(value_col).alias("count_readings")
    ).orderBy("month")
    
    monthly_patterns.show(12)
    
    return df_with_time



📂 Loading cleaned data from Day 2...
✅ Cleaned datasets loaded successfully
✅ Cleaned datasets loaded successfully



📂 Loading cleaned data from Day 2...
✅ Cleaned datasets loaded successfully
✅ Cleaned datasets loaded successfully


In [68]:
# Traffic patterns
if 'traffic' in datasets and datasets['traffic'] is not None:
    print("🚗 TRAFFIC TEMPORAL ANALYSIS")
    print("=" * 40)
    
    # Analyze vehicle count patterns
    traffic_temporal = analyze_temporal_patterns(
        datasets['traffic'], 
        'vehicle_count'
    )
    temporal_results['traffic_vehicle_count'] = traffic_temporal
    
    # TODO: Analyze speed patterns
    if 'avg_speed' in datasets['traffic'].columns:
        speed_temporal = analyze_temporal_patterns(
            datasets['traffic'],
            'avg_speed'
        )
        temporal_results['traffic_speed'] = speed_temporal

🚗 TRAFFIC TEMPORAL ANALYSIS

📈 Temporal Pattern Analysis: vehicle_count
----------------------------------------
🕐 Hourly Patterns:
+----+------------------+------------------+---------+---------+--------------+
|hour|         avg_value|      stddev_value|min_value|max_value|count_readings|
+----+------------------+------------------+---------+---------+--------------+
|   0| 17.75261904761905| 9.958785886590546|        0|       69|          4200|
|   1|17.954761904761906| 9.913214804463985|        0|       67|          4200|
|   2|17.827380952380953| 9.728217296218137|        0|       70|          4200|
|   3|17.911190476190477|10.075446210891293|        0|       65|          4200|
|   4|17.669761904761906| 9.848470417029716|        0|       70|          4200|
|   5|17.885238095238094| 9.979171983918134|        0|       69|          4200|
|   6|17.898333333333333| 9.757473285082838|        0|       60|          4200|
|   7|30.725952380952382|15.223441544781117|        0|       85|    

In [69]:
# TODO: Weather patterns
if 'weather' in datasets and datasets['weather'] is not None:
    print("\n🌤️ WEATHER TEMPORAL ANALYSIS") 
    print("=" * 40)
    
    # Analyze temperature patterns
    if 'temperature' in datasets['weather'].columns:
        weather_temporal = analyze_temporal_patterns(
            datasets['weather'],
            'temperature'
        )
        temporal_results['weather_temperature'] = weather_temporal



🌤️ WEATHER TEMPORAL ANALYSIS

📈 Temporal Pattern Analysis: temperature
----------------------------------------
🕐 Hourly Patterns:
+----+------------------+------------------+------------------+------------------+--------------+
|hour|         avg_value|      stddev_value|         min_value|         max_value|count_readings|
+----+------------------+------------------+------------------+------------------+--------------+
|   0|19.518256983018308| 3.935206913921211|10.371253989987006| 26.58534455820361|           140|
|   1|18.754363306529466|2.7205536391248812|11.971605695789275| 24.95953245196788|           140|
|   2|18.778944325686975|2.7384805307712434|12.734691825437702|26.115507675033776|           140|
|   3|18.171641491078084| 2.243416737534659|12.148398101686706| 24.09965885109917|           140|
|   4|19.526618371428793|3.4776048945257214|11.684772413395175|29.084463802869756|           140|
|   5|19.353348923333595| 2.952136441824609|14.293657303752292|26.674876025375386|  

# =============================================================================
# TODO 1.2: Pattern Anomaly Detection (60 minutes)
# =============================================================================

In [70]:
# TODO: Detect pattern anomalies in all temporal data
anomaly_results = {}

if 'traffic' in temporal_results:
    print("🚗 TRAFFIC PATTERN ANOMALY DETECTION")
    print("=" * 40)
    traffic_with_anomalies = detect_pattern_anomalies(
        temporal_results['traffic_vehicle_count'],
        'vehicle_count'
    )
    anomaly_results['traffic'] = traffic_with_anomalies
    
    # Show some anomalous patterns
    if 'vehicle_count_anomaly_daily' in traffic_with_anomalies.columns:
        print("\n🚨 Sample Daily Anomalies in Traffic:")
        anomalies = traffic_with_anomalies.filter(
            F.col("vehicle_count_anomaly_daily") == True
        ).select("timestamp", "vehicle_count", "vehicle_count_expected_daily", "hour", "day_of_week")
        anomalies.show(10)

# Detect anomalies in air quality data
if 'air_quality_pm25' in temporal_results:
    print("\n🌫️ AIR QUALITY PATTERN ANOMALY DETECTION")
    print("=" * 40)
    air_with_anomalies = detect_pattern_anomalies(
        temporal_results['air_quality_pm25'],
        'pm25'
    )
    anomaly_results['air_quality'] = air_with_anomalies

# Detect anomalies in weather data  
if 'weather_temperature' in temporal_results:
    print("\n🌤️ WEATHER PATTERN ANOMALY DETECTION")
    print("=" * 40)
    weather_with_anomalies = detect_pattern_anomalies(
        temporal_results['weather_temperature'],
        'temperature'
    )
    anomaly_results['weather'] = weather_with_anomalies

# Detect anomalies in energy data
if 'energy_power' in temporal_results:
    print("\n⚡ ENERGY PATTERN ANOMALY DETECTION")
    print("=" * 40)
    energy_with_anomalies = detect_pattern_anomalies(
        temporal_results['energy_power'],
        'power_consumption'
    )
    anomaly_results['energy'] = energy_with_anomalies

print(f"\n✅ Pattern anomaly detection completed for {len(anomaly_results)} sensor types")


🌫️ AIR QUALITY PATTERN ANOMALY DETECTION

🚨 Pattern Anomaly Detection: pm25
-----------------------------------
   📊 Daily pattern anomalies: 3092
   📊 Weekly pattern anomalies: 3113

🌤️ WEATHER PATTERN ANOMALY DETECTION

🚨 Pattern Anomaly Detection: temperature
-----------------------------------
   📊 Daily pattern anomalies: 793
   📊 Weekly pattern anomalies: 605

⚡ ENERGY PATTERN ANOMALY DETECTION

🚨 Pattern Anomaly Detection: power_consumption
-----------------------------------
   📊 Daily pattern anomalies: 36818
   📊 Weekly pattern anomalies: 35249

✅ Pattern anomaly detection completed for 3 sensor types


In [51]:
print("\n" + "=" * 60)
print("🔗 SECTION 2: CROSS-SENSOR CORRELATION ANALYSIS")
print("=" * 60)


🔗 SECTION 2: CROSS-SENSOR CORRELATION ANALYSIS


In [52]:
"""
🎯 TASK: Calculate correlations between different sensor types
💡 HINT: Join datasets on timestamp and location for meaningful correlations
📚 CONCEPTS: Correlation analysis, data fusion, causal relationships
"""

def prepare_correlation_dataset(datasets):
    """
    Prepare a combined dataset for correlation analysis
    
    Args:
        datasets: Dictionary of sensor DataFrames
    
    Returns:
        Combined DataFrame ready for correlation analysis
    """
    print("\n🔄 Preparing combined dataset for correlation analysis...")
    
    # TODO: Create a common time grid (hourly aggregations)
    # Start with traffic data as base
    if 'traffic' not in datasets:
        print("❌ Traffic data not available for correlation analysis")
        return None
        
    base_df = datasets['traffic']
    
    # TODO: Aggregate traffic data to hourly level
    traffic_hourly = base_df.withColumn("hour_timestamp", 
                                       F.date_trunc("hour", "timestamp")) \
                           .groupBy("hour_timestamp") \
                           .agg(
                               F.avg("vehicle_count").alias("avg_vehicle_count"),
                               F.avg("avg_speed").alias("avg_traffic_speed"),
                               F.count("*").alias("traffic_readings")
                           )
    
    combined_df = traffic_hourly
    
    # TODO: Add air quality data
    if 'air_quality' in datasets and datasets['air_quality'] is not None:
        air_hourly = datasets['air_quality'].withColumn("hour_timestamp",
                                                        F.date_trunc("hour", "timestamp")) \
                                            .groupBy("hour_timestamp") \
                                            .agg(
                                                F.avg("pm25").alias("avg_pm25"),
                                                F.avg("no2").alias("avg_no2"),
                                                F.avg("temperature").alias("avg_air_temp"),
                                                F.count("*").alias("air_readings")
                                            )
        
        combined_df = combined_df.join(air_hourly, "hour_timestamp", "outer")
    
    # TODO: Add weather data
    if 'weather' in datasets and datasets['weather'] is not None:
        weather_hourly = datasets['weather'].withColumn("hour_timestamp",
                                                        F.date_trunc("hour", "timestamp")) \
                                           .groupBy("hour_timestamp") \
                                           .agg(
                                               F.avg("temperature").alias("avg_weather_temp"),
                                               F.avg("humidity").alias("avg_humidity"),
                                               F.avg("wind_speed").alias("avg_wind_speed"),
                                               F.avg("precipitation").alias("avg_precipitation"),
                                               F.count("*").alias("weather_readings")
                                           )
        
        combined_df = combined_df.join(weather_hourly, "hour_timestamp", "outer")
    
    # TODO: Add energy data
    if 'energy' in datasets and datasets['energy'] is not None:
        energy_hourly = datasets['energy'].withColumn("hour_timestamp",
                                                     F.date_trunc("hour", "timestamp")) \
                                         .groupBy("hour_timestamp") \
                                         .agg(
                                             F.avg("power_consumption").alias("avg_power_consumption"),
                                             F.count("*").alias("energy_readings")
                                         )
        
        combined_df = combined_df.join(energy_hourly, "hour_timestamp", "outer")
    
    # TODO: Add time-based features
    combined_df = combined_df.withColumn("hour", F.hour("hour_timestamp")) \
                           .withColumn("day_of_week", F.dayofweek("hour_timestamp")) \
                           .withColumn("is_weekend", F.when(F.dayofweek("hour_timestamp").isin([1, 7]), True).otherwise(False))
    
    print(f"✅ Combined dataset created with {combined_df.count()} hourly records")
    return combined_df


In [53]:
def calculate_sensor_correlations(df):
    """
    Calculate correlation matrix between sensor measurements
    
    Args:
        df: Combined DataFrame with all sensor data
    
    Returns:
        Correlation matrix results
    """
    print("\n🧮 Calculating Cross-Sensor Correlations")
    print("-" * 40)
    
    # TODO: Select numeric columns for correlation
    numeric_cols = [
        "avg_vehicle_count", "avg_traffic_speed", 
        "avg_pm25", "avg_no2", "avg_air_temp",
        "avg_weather_temp", "avg_humidity", "avg_wind_speed", "avg_precipitation",
        "avg_power_consumption"
    ]
    
    # Filter to only existing columns
    available_cols = [col for col in numeric_cols if col in df.columns]
    print(f"📋 Analyzing correlations for: {available_cols}")
    
    if len(available_cols) < 2:
        print("❌ Not enough numeric columns for correlation analysis")
        return None
    
    # TODO: Prepare data for correlation calculation
    # Remove rows with null values
    df_clean = df.select(["hour_timestamp"] + available_cols).na.drop()
    
    if df_clean.count() == 0:
        print("❌ No complete records for correlation analysis")
        return None
    
    # TODO: Use Spark ML's correlation function
    # First, assemble features into a vector
    assembler = VectorAssembler(inputCols=available_cols, outputCol="features")
    df_vector = assembler.transform(df_clean)
    
    # Calculate correlation matrix
    correlation_matrix = Correlation.corr(df_vector, "features", "pearson").head()
    correlation_array = correlation_matrix[0].toArray()
    
    # TODO: Display correlation results
    print("\n📊 Correlation Matrix:")
    print("=" * 60)
    
    # Create a readable correlation matrix
    correlation_results = {}
    for i, col1 in enumerate(available_cols):
        for j, col2 in enumerate(available_cols):
            if i < j:  # Only show upper triangle
                corr_val = correlation_array[i][j]
                correlation_results[f"{col1}_vs_{col2}"] = corr_val
                print(f"{col1} vs {col2}: {corr_val:.3f}")
    
    # TODO: Identify strongest correlations
    print("\n🔝 Strongest Correlations:")
    import builtins
    sorted_correlations = sorted(correlation_results.items(), key=lambda x: builtins.abs(x[1]), reverse=True)
    for pair, corr in sorted_correlations[:5]:
        strength = "Strong" if builtins.abs(corr) > 0.7 else "Moderate" if builtins.abs(corr) > 0.5 else "Weak"
        print(f"   {pair}: {corr:.3f} ({strength})")
    
    return correlation_results

# TODO: Calculate correlations
if combined_data is not None:
    correlations = calculate_sensor_correlations(combined_data)


🚗 TRAFFIC TEMPORAL ANALYSIS

📈 Temporal Pattern Analysis: vehicle_count
----------------------------------------
🕐 Hourly Patterns:
🕐 Hourly Patterns:
+----+------------------+------------------+---------+---------+--------------+
|hour|         avg_value|      stddev_value|min_value|max_value|count_readings|
+----+------------------+------------------+---------+---------+--------------+
|   0| 17.75261904761905| 9.958785886590546|        0|       69|          4200|
|   1|17.954761904761906| 9.913214804463985|        0|       67|          4200|
|   2|17.827380952380953| 9.728217296218137|        0|       70|          4200|
|   3|17.911190476190477|10.075446210891293|        0|       65|          4200|
|   4|17.669761904761906| 9.848470417029716|        0|       70|          4200|
|   5|17.885238095238094| 9.979171983918134|        0|       69|          4200|
|   6|17.898333333333333| 9.757473285082838|        0|       60|          4200|
|   7|30.725952380952382|15.223441544781117|     

In [54]:
"""
🎯 TASK: Analyze correlations between nearby sensors
💡 HINT: Sensors close to each other should show similar patterns
📚 CONCEPTS: Spatial correlation, distance calculations, geographic clustering
"""

def analyze_spatial_correlations(df, sensor_type, max_distance_km=2.0):
    """
    Analyze correlations between spatially close sensors
    
    Args:
        df: DataFrame with sensor data including location
        sensor_type: Type of sensors to analyze
        max_distance_km: Maximum distance for correlation analysis
    
    Returns:
        Spatial correlation results
    """
    print(f"\n🗺️ Spatial Correlation Analysis: {sensor_type}")
    print("-" * 40)
    
    if 'location_lat' not in df.columns or 'location_lon' not in df.columns:
        print("❌ Location columns not found")
        return None
    
    # TODO: Get unique sensor locations
    sensor_locations = df.select("sensor_id", "location_lat", "location_lon").distinct()
    
    if sensor_locations.count() < 2:
        print("❌ Not enough sensors for spatial correlation")
        return None
    
    # TODO: For simplicity, we'll analyze a sample of sensor pairs
    # In practice, you'd calculate distances between all sensor pairs
    
    # Get list of sensors
    sensors_list = sensor_locations.collect()
    print(f"📍 Analyzing {len(sensors_list)} sensors")
    
    # TODO: Calculate distance between sensors (simplified)
    def haversine_distance(lat1, lon1, lat2, lon2):
        """Calculate distance between two points on Earth"""
        from math import radians, cos, sin, asin, sqrt
        
        # Convert to radians
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        
        # Haversine formula
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * asin(sqrt(a))
        r = 6371  # Radius of earth in kilometers
        return c * r
    
    # TODO: Find sensor pairs within max_distance_km
    nearby_pairs = []
    for i, sensor1 in enumerate(sensors_list):
        for j, sensor2 in enumerate(sensors_list[i+1:], i+1):
            distance = haversine_distance(
                sensor1['location_lat'], sensor1['location_lon'],
                sensor2['location_lat'], sensor2['location_lon']
            )
            
            if distance <= max_distance_km:
                nearby_pairs.append({
                    'sensor1': sensor1['sensor_id'],
                    'sensor2': sensor2['sensor_id'],
                    'distance_km': distance
                })
    
    print(f"🔍 Found {len(nearby_pairs)} sensor pairs within {max_distance_km}km")
    
    # TODO: Calculate correlations for nearby sensor pairs
    if len(nearby_pairs) > 0:
        print("\n📊 Spatial Correlation Results:")
        
        # For demonstration, analyze first few pairs
        for pair in nearby_pairs[:3]:
            sensor1_id = pair['sensor1'] 
            sensor2_id = pair['sensor2']
            distance = pair['distance_km']
            
            # TODO: Get time series data for both sensors
            sensor1_data = df.filter(F.col("sensor_id") == sensor1_id) \
                            .select("timestamp", "vehicle_count") \
                            .withColumnRenamed("vehicle_count", "sensor1_value")
            
            sensor2_data = df.filter(F.col("sensor_id") == sensor2_id) \
                            .select("timestamp", "vehicle_count") \
                            .withColumnRenamed("vehicle_count", "sensor2_value")
            
            # Join on timestamp
            paired_data = sensor1_data.join(sensor2_data, "timestamp", "inner")
            
            if paired_data.count() > 10:  # Need enough data points
                # Calculate correlation using Spark ML
                assembler = VectorAssembler(
                    inputCols=["sensor1_value", "sensor2_value"], 
                    outputCol="features"
                )
                paired_vector = assembler.transform(paired_data.na.drop())
                
                if paired_vector.count() > 0:
                    corr_matrix = Correlation.corr(paired_vector, "features", "pearson").head()
                    correlation = corr_matrix[0].toArray()[0][1]
                    
                    print(f"   {sensor1_id} vs {sensor2_id}: {correlation:.3f} (distance: {distance:.2f}km)")
    
    return nearby_pairs



🌫️ AIR QUALITY TEMPORAL ANALYSIS

📈 Temporal Pattern Analysis: pm25
----------------------------------------
🕐 Hourly Patterns:



🌫️ AIR QUALITY TEMPORAL ANALYSIS

📈 Temporal Pattern Analysis: pm25
----------------------------------------
🕐 Hourly Patterns:


+----+------------------+-----------------+-------------------+------------------+--------------+
|hour|         avg_value|     stddev_value|          min_value|         max_value|count_readings|
+----+------------------+-----------------+-------------------+------------------+--------------+
|   0|25.362712497175856|8.232537410233412|                0.0| 50.94222176264096|           560|
|   1|24.959245415496294|8.238518804871239|                0.0| 46.73358328000002|           560|
|   2| 24.66777158925574|7.847364431326072| 3.7993009109286895| 46.44713601667584|           560|
|   3|24.648206274701522|7.904155756441992| 1.0536003340013522| 47.88329961552654|           560|
|   4|24.678668973889884|7.837281376057234|                0.0| 49.10744210159358|           560|
|   5|24.982470454927267|8.113342987376775| 0.6804415678177129|58.762627775831916|           560|
|   6|24.801239371050865|7.668919312343141|                0.0|45.970611061432166|           560|
|   7| 32.2164529733

# =============================================================================
# SECTION 3: FEATURE ENGINEERING (Afternoon - 3 hours)
# =============================================================================

# =============================================================================
# TODO 3.1: Lag Features Creation (60 minutes)
# =============================================================================

In [23]:
# TODO: Create lag features for traffic data
if 'traffic' in datasets:
    print("🚗 Creating lag features for traffic data...")
    traffic_with_lags = create_lag_features(
        datasets['traffic'],
        ['vehicle_count', 'avg_speed'],
        lag_periods=[1, 6, 12, 24],  # 1, 6, 12, 24 periods back
        sensor_col='sensor_id'
    )
    
    # Show sample of lag features
    print("\n📊 Sample Lag Features:")
    lag_cols = ['timestamp', 'sensor_id', 'vehicle_count', 'vehicle_count_lag_1', 
               'vehicle_count_lag_6', 'vehicle_count_diff_1', 'vehicle_count_pct_change_1']
    available_cols = [col for col in lag_cols if col in traffic_with_lags.columns]
    traffic_with_lags.select(available_cols).show(10)


🚗 Creating lag features for traffic data...

🔄 Creating Lag Features
-------------------------
   Creating lags for vehicle_count...
      vehicle_count_lag_1: 100.0% availability
      vehicle_count_lag_6: 99.7% availability
      vehicle_count_lag_12: 99.4% availability
      vehicle_count_lag_24: 98.8% availability
   Creating lags for avg_speed...
      avg_speed_lag_1: 100.0% availability
      avg_speed_lag_6: 99.7% availability
      avg_speed_lag_12: 99.4% availability
      avg_speed_lag_24: 98.8% availability
✅ Lag features created for 2 columns

📊 Sample Lag Features:
+--------------------+-----------+-------------+-------------------+-------------------+--------------------+--------------------------+
|           timestamp|  sensor_id|vehicle_count|vehicle_count_lag_1|vehicle_count_lag_6|vehicle_count_diff_1|vehicle_count_pct_change_1|
+--------------------+-----------+-------------+-------------------+-------------------+--------------------+--------------------------+
|20

In [57]:
"""
🎯 TASK: Calculate rolling statistics for feature engineering
💡 HINT: Moving averages smooth out noise and show trends
📚 CONCEPTS: Moving averages, rolling windows, trend indicators
"""

def create_rolling_features(df, value_columns, windows=[6, 12, 24, 48], time_col="timestamp", sensor_col=None):
    """
    Calculate rolling statistics for feature engineering
    
    Args:
        df: DataFrame with time series data
        value_columns: List of columns to calculate rolling stats for
        windows: List of window sizes for rolling calculations
        time_col: Timestamp column
        sensor_col: Sensor ID column for per-sensor calculations
    
    Returns:
        DataFrame with rolling features added
    """
    print(f"\n📊 Creating Rolling Statistics Features")
    print("-" * 35)
    
    result_df = df
    
    # TODO: Create window specifications for different rolling periods
    if sensor_col:
        base_window = Window.partitionBy(sensor_col).orderBy(time_col)
    else:
        base_window = Window.orderBy(time_col)
    
    # TODO: Create rolling features for each column and window size
    for col in value_columns:
        if col not in df.columns:
            continue
            
        print(f"   Creating rolling features for {col}...")
        
        for window_size in windows:
            # TODO: Rolling mean
            rolling_mean_col = f"{col}_rolling_mean_{window_size}"
            window_spec = base_window.rowsBetween(-window_size + 1, 0)
            result_df = result_df.withColumn(
                rolling_mean_col,
                F.avg(col).over(window_spec)
            )
            
            # TODO: Rolling standard deviation
            rolling_std_col = f"{col}_rolling_std_{window_size}"
            result_df = result_df.withColumn(
                rolling_std_col,
                F.stddev(col).over(window_spec)
            )
            
            # TODO: Rolling min and max
            rolling_min_col = f"{col}_rolling_min_{window_size}"
            rolling_max_col = f"{col}_rolling_max_{window_size}"
            result_df = result_df.withColumn(rolling_min_col, F.min(col).over(window_spec)) \
                               .withColumn(rolling_max_col, F.max(col).over(window_spec))
            
            # TODO: Rolling range
            rolling_range_col = f"{col}_rolling_range_{window_size}"
            result_df = result_df.withColumn(
                rolling_range_col,
                F.col(rolling_max_col) - F.col(rolling_min_col)
            )
            
            # TODO: Position within rolling window (current value relative to min/max)
            position_col = f"{col}_position_in_window_{window_size}"
            result_df = result_df.withColumn(
                position_col,
                F.when(F.col(rolling_range_col) > 0,
                       (F.col(col) - F.col(rolling_min_col)) / F.col(rolling_range_col)
                ).otherwise(0.5)  # If no range, assume middle position
            )
            
            print(f"      Window {window_size}: mean, std, min, max, range, position")
    
    print(f"✅ Rolling features created for {len(value_columns)} columns")
    return result_df


# =============================================================================
# TODO 3.3: Interaction Features (60 minutes)
# =============================================================================

In [27]:
# TODO: Create interaction features for all sensor types
feature_datasets = {}

for name, df in datasets.items():
    if df is not None and name != 'zones':
        try:
            df_with_interactions = create_interaction_features(df, name)
            feature_datasets[name] = df_with_interactions
            print(f"✅ Interaction features created for {name}")
        except Exception as e:
            print(f"❌ Error creating interactions for {name}: {str(e)}")


🔗 Creating Interaction Features: traffic
----------------------------------------
🔗 Interaction features created: ['traffic_flow', 'congestion_indicator', 'speed_efficiency']
✅ Interaction features created for traffic

🔗 Creating Interaction Features: air_quality
----------------------------------------
🔗 Interaction features created: ['combined_pollution', 'temp_adjusted_pm25']
✅ Interaction features created for air_quality

🔗 Creating Interaction Features: weather
----------------------------------------
🔗 Interaction features created: ['heat_index', 'wind_chill']
✅ Interaction features created for weather

🔗 Creating Interaction Features: energy
----------------------------------------
🔗 Interaction features created: ['calculated_power_factor', 'building_efficiency']
✅ Interaction features created for energy


In [59]:
print("\n" + "=" * 60)
print("📈 SECTION 4: TREND ANALYSIS")
print("=" * 60)


🔗 SECTION 2: CROSS-SENSOR CORRELATION ANALYSIS


In [29]:
"""
🎯 TASK: Identify and quantify trends in sensor data
💡 HINT: Use statistical methods to detect significant trends
📚 CONCEPTS: Trend analysis, linear regression, changepoint detection
"""

def detect_and_quantify_trends(df, value_col, time_col="timestamp", sensor_col=None):
    """
    Identify and quantify trends in sensor data
    
    Args:
        df: DataFrame with time series data
        value_col: Column to analyze for trends
        time_col: Timestamp column
        sensor_col: Sensor ID column for per-sensor analysis
    
    Returns:
        DataFrame with trend indicators
    """
    print(f"\n📈 Trend Detection: {value_col}")
    print("-" * 25)
    
    # TODO: Add time-based numeric features for trend analysis
    df_with_time_numeric = df.withColumn(
        "time_numeric",
        F.unix_timestamp(time_col)
    ).withColumn(
        "day_numeric", 
        F.datediff(F.col(time_col), F.lit("2024-01-01"))
    )
    
    # TODO: Calculate overall trend using simple linear regression approach
    # For each sensor (if sensor_col provided) or globally
    
    if sensor_col:
        # Per-sensor trend analysis
        print("   Calculating per-sensor trends...")
        
        # TODO: Use window functions to calculate trend indicators
        window_spec = Window.partitionBy(sensor_col).orderBy(time_col)
        
        # Calculate moving trend (slope over rolling window)
        trend_window = window_spec.rowsBetween(-23, 0)  # 24-point window
        
        # Simple trend indicator: correlation between time and value
        df_with_trends = df_with_time_numeric.withColumn(
            f"{value_col}_trend_direction",
            # Simplified trend: compare current value to value 24 periods ago
            F.when(
                F.col(value_col) > F.lag(value_col, 24).over(window_spec),
                1  # Upward trend
            ).when(
                F.col(value_col) < F.lag(value_col, 24).over(window_spec),
                -1  # Downward trend
            ).otherwise(0)  # No clear trend
        )
        
        # TODO: Calculate trend strength
        # Use rolling standard deviation to measure volatility
        df_with_trends = df_with_trends.withColumn(
            f"{value_col}_trend_strength",
            F.abs(
                (F.col(value_col) - F.lag(value_col, 24).over(window_spec)) / 
                (F.stddev(value_col).over(trend_window) + 0.001)  # Avoid division by zero
            )
        )
        
    else:
        # Global trend analysis
        print("   Calculating global trends...")
        
        # TODO: Simple global trend using window functions
        window_spec = Window.orderBy(time_col)
        
        df_with_trends = df_with_time_numeric.withColumn(
            f"{value_col}_trend_direction",
            F.when(
                F.col(value_col) > F.lag(value_col, 24).over(window_spec),
                1
            ).when(
                F.col(value_col) < F.lag(value_col, 24).over(window_spec),
                -1
            ).otherwise(0)
        )
    
    # TODO: Identify changepoints (simplified approach)
    # Look for significant changes in rolling mean
    if sensor_col:
        window_before = Window.partitionBy(sensor_col).orderBy(time_col).rowsBetween(-11, -1)
        window_after = Window.partitionBy(sensor_col).orderBy(time_col).rowsBetween(1, 11)
    else:
        window_before = Window.orderBy(time_col).rowsBetween(-11, -1)
        window_after = Window.orderBy(time_col).rowsBetween(1, 11)
    
    df_with_trends = df_with_trends.withColumn(
        f"{value_col}_changepoint_indicator",
        F.abs(
            F.avg(value_col).over(window_after) - F.avg(value_col).over(window_before)
        ) / (F.stddev(value_col).over(window_before) + 0.001)
    )
    
    # TODO: Flag significant changepoints
    df_with_trends = df_with_trends.withColumn(
        f"{value_col}_significant_changepoint",
        F.when(F.col(f"{value_col}_changepoint_indicator") > 2.0, True).otherwise(False)
    )
    
    # TODO: Summarize trend findings
    if sensor_col:
        trend_summary = df_with_trends.groupBy(sensor_col).agg(
            F.avg(f"{value_col}_trend_direction").alias("avg_trend_direction"),
            F.avg(f"{value_col}_trend_strength").alias("avg_trend_strength"),
            F.sum(F.when(F.col(f"{value_col}_significant_changepoint"), 1).otherwise(0)).alias("changepoint_count")
        )
        
        print("   📊 Trend Summary by Sensor:")
        trend_summary.show(10)
    else:
        # Global trend summary
        overall_trend = df_with_trends.agg(
            F.avg(f"{value_col}_trend_direction").alias("overall_trend_direction"),
            F.avg(f"{value_col}_trend_strength").alias("overall_trend_strength")
        ).collect()[0]
        
        print(f"   📊 Overall Trend Direction: {overall_trend['overall_trend_direction']:.3f}")
        print(f"   📊 Overall Trend Strength: {overall_trend['overall_trend_strength']:.3f}")
    
    return df_with_trends


# =============================================================================
# DAY 3 DELIVERABLES & VALIDATION
# =============================================================================

In [72]:
def validate_day3_completion():
    """Validate that Day 3 objectives have been met"""
    
    checklist = {
        "temporal_patterns_analyzed": False,
        "pattern_anomalies_detected": False,
        "correlation_analysis_completed": False,
        "spatial_correlations_analyzed": False,
        "lag_features_created": False,
        "rolling_features_created": False,
        "interaction_features_engineered": False,
        "trend_analysis_completed": False,
        "feature_pipeline_documented": False
    }
    
    try:
        # Check temporal pattern analysis
        if 'temporal_results' in globals() and len(temporal_results) > 0:
            checklist["temporal_patterns_analyzed"] = True
            
        # Check pattern anomaly detection - UPDATED TO PROPERLY DETECT
        if 'anomaly_results' in globals() and len(anomaly_results) > 0:
            checklist["pattern_anomalies_detected"] = True
            
        # Check correlation analysis
        if 'correlations' in globals() and correlations:
            checklist["correlation_analysis_completed"] = True
            
        # Check spatial correlation analysis
        if 'spatial_correlations' in globals():
            checklist["spatial_correlations_analyzed"] = True
            
        # Check feature engineering
        if 'feature_datasets' in globals() and len(feature_datasets) > 0:
            checklist["interaction_features_engineered"] = True
            
        # Check if lag/rolling features were created
        if 'traffic_with_lags' in globals():
            checklist["lag_features_created"] = True

        if 'traffic_with_rolling' in globals():
            checklist["rolling_features_created"] = True
            
        # Check trend analysis
        if 'traffic_trends' in globals():
            checklist["trend_analysis_completed"] = True
            
        # Check feature pipeline documentation - UPDATED TO PROPERLY DETECT
        if 'pipeline_documentation' in globals() or 'feature_pipeline_doc' in globals():
            checklist["feature_pipeline_documented"] = True
        
    except Exception as e:
        print(f"❌ Validation error: {str(e)}")
    
    # Display results
    print("✅ COMPLETION STATUS:")
    for item, status in checklist.items():
        status_icon = "✅" if status else "❌"
        print(f"   {status_icon} {item.replace('_', ' ').title()}")
    
    import builtins
    completion_rate = builtins.sum(checklist.values()) / len(checklist) * 100
    print(f"\n📊 Overall Completion: {completion_rate:.1f}%")
    
    if completion_rate >= 70:
        print("🎉 Excellent work! You're ready for Day 4!")
        print("\n📈 KEY INSIGHTS FROM DAY 3:")
        print("- Temporal patterns reveal operational insights")
        print("- Cross-sensor correlations show system interconnections")
        print("- Engineered features capture domain knowledge")
        print("- Trend analysis identifies long-term changes")
    else:
        print("📝 Please review incomplete items before proceeding to Day 4.")
    
    return checklist

# Run validation
completion_status = validate_day3_completion()

✅ COMPLETION STATUS:
   ✅ Temporal Patterns Analyzed
   ✅ Pattern Anomalies Detected
   ✅ Correlation Analysis Completed
   ✅ Spatial Correlations Analyzed
   ✅ Lag Features Created
   ✅ Rolling Features Created
   ✅ Interaction Features Engineered
   ✅ Trend Analysis Completed
   ❌ Feature Pipeline Documented

📊 Overall Completion: 88.9%
🎉 Excellent work! You're ready for Day 4!

📈 KEY INSIGHTS FROM DAY 3:
- Temporal patterns reveal operational insights
- Cross-sensor correlations show system interconnections
- Engineered features capture domain knowledge
- Trend analysis identifies long-term changes


In [62]:
print("\n💾 SAVING ENGINEERED FEATURES FOR DAY 4")
print("=" * 40)


🧮 Calculating Cross-Sensor Correlations
----------------------------------------
📋 Analyzing correlations for: ['avg_vehicle_count', 'avg_traffic_speed', 'avg_pm25', 'avg_no2', 'avg_air_temp', 'avg_weather_temp', 'avg_humidity', 'avg_wind_speed', 'avg_precipitation', 'avg_power_consumption']



🧮 Calculating Cross-Sensor Correlations
----------------------------------------
📋 Analyzing correlations for: ['avg_vehicle_count', 'avg_traffic_speed', 'avg_pm25', 'avg_no2', 'avg_air_temp', 'avg_weather_temp', 'avg_humidity', 'avg_wind_speed', 'avg_precipitation', 'avg_power_consumption']



🧮 Calculating Cross-Sensor Correlations
----------------------------------------
📋 Analyzing correlations for: ['avg_vehicle_count', 'avg_traffic_speed', 'avg_pm25', 'avg_no2', 'avg_air_temp', 'avg_weather_temp', 'avg_humidity', 'avg_wind_speed', 'avg_precipitation', 'avg_power_consumption']



📊 Correlation Matrix:
avg_vehicle_count vs avg_traffic_speed: -0.997
avg_vehicle_count vs avg_pm25: 0.786
avg_vehicle_count vs avg_no2: 0.765
avg_vehicle_count vs avg_air_temp: -0.048
avg_vehicle_count vs avg_weather_temp: 0.083
avg_vehicle_count vs avg_humidity: -0.018
avg_vehicle_count vs avg_wind_speed: -0.014
avg_vehicle_count vs avg_precipitation: -0.108
avg_vehicle_count vs avg_power_consumption: 0.341
avg_traffic_speed vs avg_pm25: -0.787
avg_traffic_speed vs avg_no2: -0.764
avg_traffic_speed vs avg_air_temp: 0.054
avg_traffic_speed vs avg_weather_temp: -0.081
avg_traffic_speed vs avg_humidity: 0.017
avg_traffic_speed vs avg_wind_speed: 0.008
avg_traffic_speed vs avg_precipitation: 0.111
avg_traffic_speed vs avg_power_consumption: -0.343
avg_pm25 vs avg_no2: 0.937
avg_pm25 vs avg_air_temp: -0.051
avg_pm25 vs avg_weather_temp: 0.071
avg_pm25 vs avg_humidity: -0.105
avg_pm25 vs avg_wind_speed: 0.024
avg_pm25 vs avg_precipitation: 0.014
avg_pm25 vs avg_power_consumption: 0.193
avg

# =============================================================================
# NEXT STEPS
# =============================================================================

In [63]:
print("""
📅 DAY 4 PREVIEW: Advanced Analytics & Anomaly Detection

Tomorrow you'll work on:
1. 🤖 Machine learning model development using engineered features
2. 🚨 Advanced anomaly detection systems
3. 📊 Predictive modeling for traffic, air quality, and energy
4. ⚡ Pipeline optimization and performance tuning
5. 🎯 Real-time analytics implementation

📚 RECOMMENDED PREPARATION:
- Review machine learning concepts and Spark MLlib
- Understand anomaly detection algorithms
- Read about model evaluation and validation
- Practice with classification and regression in Spark

💡 KEY TAKEAWAYS FROM DAY 3:
- Time series patterns reveal operational rhythms and anomalies
- Cross-sensor correlations indicate system-wide relationships
- Lag and rolling features capture temporal dependencies
- Interaction features encode domain expertise
- Trend analysis identifies long-term operational changes

🤝 QUESTIONS FOR REFLECTION:
- Which temporal patterns were most surprising?
- What correlations suggest causal relationships?
- How might seasonal factors affect your feature engineering?
- Which engineered features seem most predictive?

🔧 FEATURE ENGINEERING SUMMARY:
""")

# TODO: Generate feature engineering summary
if len(feature_datasets) > 0:
    print("📊 Features Created by Dataset:")
    for name, df in feature_datasets.items():
        feature_cols = [col for col in df.columns if any(keyword in col for keyword in 
                       ['lag_', 'rolling_', '_diff_', '_pct_change_', '_trend_', '_flow', '_indicator'])]
        print(f"   {name}: {len(feature_cols)} engineered features")

print("\n💾 Don't forget to save your notebook and commit your changes!")



📅 DAY 4 PREVIEW: Advanced Analytics & Anomaly Detection

Tomorrow you'll work on:
1. 🤖 Machine learning model development using engineered features
2. 🚨 Advanced anomaly detection systems
3. 📊 Predictive modeling for traffic, air quality, and energy
4. ⚡ Pipeline optimization and performance tuning
5. 🎯 Real-time analytics implementation

📚 RECOMMENDED PREPARATION:
- Review machine learning concepts and Spark MLlib
- Understand anomaly detection algorithms
- Read about model evaluation and validation
- Practice with classification and regression in Spark

💡 KEY TAKEAWAYS FROM DAY 3:
- Time series patterns reveal operational rhythms and anomalies
- Cross-sensor correlations indicate system-wide relationships
- Lag and rolling features capture temporal dependencies
- Interaction features encode domain expertise
- Trend analysis identifies long-term operational changes

🤝 QUESTIONS FOR REFLECTION:
- Which temporal patterns were most surprising?
- What correlations suggest causal relation